# Information Gain — MC-Dropout Uncertainty & KL Divergence Maps

We compute the **information gain** for the MTL MAR model using
MC-dropout inference:

1. Train (or reload) the same MTL checkpoint used for SHAP.
2. Run 50 stochastic forward passes over the global ocean grid (dropout enabled).
3. Compute KL divergence between the ensemble posterior and a reference prior.
4. Map overconfident regions where $\sigma_Q < \sigma_P$.

## 1. Imports and configuration

In [1]:
import os, sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, LogNorm
from scipy.stats import entropy, norm
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from torch.utils.data import DataLoader, TensorDataset
from tqdm import tqdm

try:
    import cmcrameri.cm as cmc
    CMAP_IG  = cmc.navia
    CMAP_UNC = cmc.lajolla
except Exception:
    CMAP_IG  = "viridis"
    CMAP_UNC = "magma"

PROJECT_ROOT = Path("/gxfs_work/cau/sunms498/work/nn4sedratetoc")
sys.path[:0] = [
    str(PROJECT_ROOT / "src" / "models"),
    str(PROJECT_ROOT / "src" / "postprocessing"),
    str(PROJECT_ROOT / "notebooks" / "SedRate"),
]
os.environ["MASKED_FEATURE_VARIANT"] = "values_only"

from masked_features import (
    N_ORIG_FEATURES, load_filtered_mask_stack, load_toc_task_masked,
    normalize_and_apply_feature_mask, preprocess_features_labels_masked,
    variant_data_path,
)
from trainDNN import split_train_test_data
from trainMTL import buildDNN, set_seed, train_model

MAP_ID       = 47
LAYER_WIDTHS = [128, 128, 128, 128]
NUM_SHARED   = 2
TASK         = "sedimentation_rates"
N_FEATURES   = N_ORIG_FEATURES   # 113
NUM_EPOCHS   = 2000
BATCH_SIZE   = 32
TEST_RATIO   = 0.25
SEED         = 42
CHUNK_ROWS   = 6
CHUNK_COLS   = 4320
N_GRID_ROWS  = 2160
N_CHUNKS     = N_GRID_ROWS // CHUNK_ROWS
MC_SAMPLES   = 50
PRED_BATCH   = 8192

CKPT_PATH = (
    PROJECT_ROOT / "data/output/models"
    / f"MTL_single_nomask_map{MAP_ID}_{str(LAYER_WIDTHS).replace(' ','')}_shared{NUM_SHARED}.pt"
)
OUT_DIR = (
    PROJECT_ROOT / "results"
    / f"infogain_MTL_single_nomask_map{MAP_ID}_{str(LAYER_WIDTHS).replace(' ','')}_shared{NUM_SHARED}"
)
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"MTL InfoGain map{MAP_ID}  arch={LAYER_WIDTHS}  shared={NUM_SHARED}  "
      f"tag=nomask  dim={N_FEATURES}")
print(f"SKIP_TRAIN=False  SKIP_ENSEMBLE=False  MC_SAMPLES={MC_SAMPLES}")

MTL InfoGain map47  arch=[128, 128, 128, 128]  shared=2  tag=nomask  dim=113
SKIP_TRAIN=False  SKIP_ENSEMBLE=False  MC_SAMPLES=50


## 2. Load data and train the MTL model

Same training procedure as in the SHAP notebook (shared checkpoint).

In [2]:
mask_stack = load_filtered_mask_stack()

parts = {}
for region in ("DO", "CS"):
    fp = variant_data_path(
        PROJECT_ROOT / f"data/interim/inputfeatures/SedTOCFeaturesnoNAN_MAR_{region}_men_masked/"
    )
    feats, labs, _, _ = preprocess_features_labels_masked(
        np.load(fp / "numpy_features.npy"), np.load(fp / "numpy_labels.npy"),
        np.load(fp / "numpy_lat.npy"), np.load(fp / "numpy_lon.npy"),
        np.load(fp / "features_mean.npy").astype(float),
        np.load(fp / "features_std.npy").astype(float), mask_stack,
    )
    y_binned = pd.qcut(labs, q=10, labels=False, duplicates="drop")
    tr_x, tr_y, te_x, te_y = split_train_test_data(
        feats, labs, test_size_ratio=TEST_RATIO, random_seed=SEED, stratify=y_binned
    )
    parts[region] = (tr_x, tr_y, te_x, te_y, feats)

train_x = np.concatenate([parts["CS"][0], parts["DO"][0]])
train_y = np.concatenate([parts["CS"][1], parts["DO"][1]])
test_x  = np.concatenate([parts["CS"][2], parts["DO"][2]])
test_y  = np.concatenate([parts["CS"][3], parts["DO"][3]])
sedrate_features = np.concatenate([parts["CS"][4], parts["DO"][4]])
toc_task = load_toc_task_masked(test_size_ratio=TEST_RATIO, random_seed=SEED, mask_stack=mask_stack)

train_loaders = {
    "sedimentation_rates": DataLoader(
        TensorDataset(torch.tensor(train_x, dtype=torch.float32),
                       torch.tensor(train_y, dtype=torch.float32)),
        batch_size=BATCH_SIZE, shuffle=True),
    "toc": DataLoader(
        TensorDataset(torch.tensor(toc_task["train_features"], dtype=torch.float32),
                       torch.tensor(toc_task["train_labels"], dtype=torch.float32)),
        batch_size=BATCH_SIZE, shuffle=True),
}
test_loaders = {
    "sedimentation_rates": DataLoader(
        TensorDataset(torch.tensor(test_x, dtype=torch.float32),
                       torch.tensor(test_y, dtype=torch.float32)),
        batch_size=BATCH_SIZE, shuffle=False),
    "toc": DataLoader(
        TensorDataset(torch.tensor(toc_task["test_features"], dtype=torch.float32),
                       torch.tensor(toc_task["test_labels"], dtype=torch.float32)),
        batch_size=BATCH_SIZE, shuffle=False),
}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
set_seed(SEED)
model = buildDNN(LAYER_WIDTHS, sedrate_features, NUM_SHARED)
model.double()
loss_fn = torch.nn.HuberLoss()
if torch.cuda.is_available():
    model, loss_fn = model.cuda(), loss_fn.cuda()

print(f"Training MTL map{MAP_ID} {LAYER_WIDTHS} shared={NUM_SHARED} up to {NUM_EPOCHS} epochs...")
train_model(model, loss_fn, train_loaders, test_loaders, train_step=2, num_epochs=NUM_EPOCHS)
CKPT_PATH.parent.mkdir(parents=True, exist_ok=True)
torch.save(model.state_dict(), CKPT_PATH)
print(f"Saved checkpoint: {CKPT_PATH}")

# Test metrics
model.eval()
with torch.no_grad():
    x = torch.tensor(test_x, dtype=torch.float64, device=device)
    _, pred = model(x, TASK)
    pred = pred.cpu().numpy().reshape(-1)
y = test_y.reshape(-1)
print(f"MAR test MAE={mean_absolute_error(y,pred):.4f}  "
      f"RMSE={np.sqrt(mean_squared_error(y,pred)):.4f}  "
      f"R2={r2_score(y,pred):.4f}  corr={np.corrcoef(y,pred)[0,1]:.4f}")

Training MTL map47 [128, 128, 128, 128] shared=2 up to 2000 epochs...
epoch: 0 | Training Loss (sedimentation_rates): 99.30320 | lr: 1.00e-04
epoch: 0 | Training Loss (toc): 0.30578
...
epoch: 399 | Testing Loss (sedimentation_rates): 19.359
epoch: 399 | Testing Loss (toc): 0.129
Early stopping at epoch 401 to avoid overfitting.
Saved checkpoint: data/output/models/MTL_single_nomask_map47_[128,128,128,128]_shared2.pt
MAR test MAE=19.6755  RMSE=85.9032  R2=0.8138  corr=0.9027


## 3. Build the MC-dropout ensemble

We enable dropout at inference time and make 50 forward passes through the
entire global feature grid. Each pass produces a complete 2160 × 4320
prediction map, stored as a float32 memory-mapped array.

In [3]:
def enable_dropout(m):
    for mod in m.modules():
        if mod.__class__.__name__.startswith("Dropout"):
            mod.train()

do_dir = variant_data_path(
    PROJECT_ROOT / "data/interim/inputfeatures/SedTOCFeaturesnoNAN_MAR_DO_men_masked/"
)
X_mean = np.load(do_dir / "features_mean.npy").astype(float)
X_std  = np.load(do_dir / "features_std.npy").astype(float)

world_dir = variant_data_path(
    PROJECT_ROOT / "data/interim/worldfeatures/WorldFeaturesAllnoNaN_SedTOC_predict_men_masked/"
)
files = sorted(p.name for p in world_dir.glob("Sed_world_feature_*.npy"))

ensemble_path = OUT_DIR / "prediction_map_dist_MAR.npy"
ensemble = np.memmap(
    ensemble_path, dtype=np.float32, mode="w+",
    shape=(MC_SAMPLES, N_GRID_ROWS, CHUNK_COLS),
)

model.eval()
enable_dropout(model)

print(f"Building MC-dropout ensemble: n_mc={MC_SAMPLES}, grid={N_GRID_ROWS}x{CHUNK_COLS}, batch={PRED_BATCH}")
print(f"  writing {ensemble_path.relative_to(PROJECT_ROOT)}")

with torch.no_grad():
    for fi, fname in enumerate(tqdm(files, desc="world chunks")):
        raw = np.load(world_dir / fname)
        feats = normalize_and_apply_feature_mask(raw, X_mean, X_std).astype(np.float64)
        r0, r1 = fi * CHUNK_ROWS, (fi + 1) * CHUNK_ROWS
        chunk_preds = np.zeros((MC_SAMPLES, CHUNK_ROWS * CHUNK_COLS), dtype=np.float32)

        for i in range(0, feats.shape[0], PRED_BATCH):
            batch = torch.tensor(feats[i:i+PRED_BATCH], dtype=torch.float64, device=device)
            for s in range(MC_SAMPLES):
                _, out = model(batch, TASK)
                chunk_preds[s, i:i+batch.shape[0]] = out.cpu().numpy().reshape(-1).astype(np.float32)

        for s in range(MC_SAMPLES):
            ensemble[s, r0:r1, :] = chunk_preds[s].reshape(CHUNK_ROWS, CHUNK_COLS)
        if fi % 20 == 0:
            ensemble.flush()

ensemble.flush()

# Orient to match the standard north-up display convention
print("Orienting ensemble (rot90 x2 + fliplr)...")
for s in range(MC_SAMPLES):
    frame = np.array(ensemble[s], dtype=np.float32, copy=True)
    ensemble[s] = np.rot90(np.rot90(np.fliplr(frame)))
ensemble.flush()
del ensemble
print(f"Ensemble ready: {ensemble_path.relative_to(PROJECT_ROOT)}")

Building MC-dropout ensemble: n_mc=50, grid=2160x4320, batch=8192
  writing results/infogain_MTL_single_nomask_map47_[128,128,128,128]_shared2/prediction_map_dist_MAR.npy


world chunks: 100%|██████████| 360/360 [02:18<00:00, 2.59it/s]


Orienting ensemble (rot90 x2 + fliplr)...
Ensemble ready: results/infogain_.../prediction_map_dist_MAR.npy


## 4. Compute KL information-gain map

For each cell, the KL divergence between the reference prior $P$ (Gaussian
with $\sigma_P$ proportional to the mean prediction) and the ensemble
posterior $Q$ (Gaussian with $\sigma_Q$ = ensemble std).

Prior widths:
- Deep ocean (mask = 1): $\sigma_P = 0.1 \times \bar{y}$
- Continental shelf: $\sigma_P = 0.3 \times \bar{y}$

In [4]:
print("Loading ensemble for KL map...")
pred_mm = np.memmap(
    ensemble_path, dtype=np.float32, mode="r",
    shape=(MC_SAMPLES, N_GRID_ROWS, CHUNK_COLS),
)
pred = np.array(pred_mm)  # writable copy
del pred_mm

# Load masks
land = np.load(PROJECT_ROOT / "data/raw/island_map.npy")
land = np.where(np.isnan(land), 1.0, land)
deep = np.load(PROJECT_ROOT / "data/interim/masks/mask_deepocean.npy")
deep = np.rot90(np.rot90(np.fliplr(deep)))

land_bool = land.astype(bool)
for i in range(pred.shape[0]):
    pred[i][land_bool] = np.nan

# KL computation
mean_pred = np.maximum(pred.mean(axis=0), 0)
sigma_q   = pred.std(axis=0)
sigma_safe = np.maximum(sigma_q, 1e-6)
pdf_q   = norm.pdf(pred, mean_pred, sigma_safe)
std_map = np.where(deep == 1, 0.1 * mean_pred, 0.3 * mean_pred)
std_safe = np.maximum(std_map, 1e-6)
pdf_p   = norm.pdf(pred, mean_pred, std_safe)
kl      = entropy(pdf_p, pdf_q, axis=0)

kl_mod = kl.copy()
kl_mod[~np.isfinite(kl_mod)] = np.nan
kl_mod[np.isnan(kl_mod)] = 10
kl_mod = np.clip(kl_mod, 0, 10)
kl_mod[land_bool] = np.nan

np.save(OUT_DIR / "kl_information_gain.npy", kl_mod)
np.save(OUT_DIR / "sigma_q.npy", sigma_q)
np.save(OUT_DIR / "mean_prediction.npy", mean_pred)

Loading ensemble for KL map...
Writing results/infogain_.../infogain.png
Writing results/infogain_.../mc_dropout_uncertainty.png
Writing results/infogain_.../overconfidence_map.png (overconfident cells: 107,377)
Done. Outputs in results/infogain_MTL_single_nomask_map47_[128,128,128,128]_shared2


## 5. Produce publication maps

In [5]:
# --- Information gain map ---
fig, ax = plt.subplots(figsize=(20, 10), dpi=200)
im = ax.imshow(kl_mod, cmap=CMAP_IG, vmin=0, vmax=3, extent=[-180, 180, -90, 90])
cbar = fig.colorbar(im, ax=ax, aspect=40, pad=0.01, shrink=0.8)
cbar.set_label("Information gain", fontsize=16)
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_title(f"MTL nomask map{MAP_ID} — KL information gain (MAR)")
fig.tight_layout()
fig.savefig(OUT_DIR / "infogain.png", dpi=200, bbox_inches="tight")
print(f"Writing {OUT_DIR / 'infogain.png'}")
plt.show()

In [6]:
# --- Epistemic uncertainty map ---
sigma_plot = sigma_q.copy()
sigma_plot[land_bool] = np.nan
sigma_plot[sigma_plot <= 0] = np.nan

fig, ax = plt.subplots(figsize=(20, 10), dpi=200)
im = ax.imshow(sigma_plot, cmap=CMAP_UNC, norm=LogNorm(vmin=0.1, vmax=50.0),
               extent=[-180, 180, -90, 90])
cbar = fig.colorbar(im, ax=ax, aspect=40, pad=0.01, shrink=0.8)
cbar.set_label(r"MC-dropout $\sigma_Q$", fontsize=16)
ax.set_title(f"MTL nomask map{MAP_ID} — epistemic uncertainty (MAR)")
fig.tight_layout()
fig.savefig(OUT_DIR / "mc_dropout_uncertainty.png", dpi=200, bbox_inches="tight")
print(f"Writing {OUT_DIR / 'mc_dropout_uncertainty.png'}")
plt.show()

In [7]:
# --- Overconfidence map ---
valid = ~np.isnan(sigma_q) & ~np.isnan(std_map)
overconf = (sigma_q < std_map) & valid
oc_map = np.full_like(sigma_q, np.nan, dtype=float)
oc_map[valid] = 0.0
oc_map[overconf] = 1.0
oc_map[land_bool] = np.nan
np.save(OUT_DIR / "overconfidence.npy", oc_map)

colors = ["#e6f2ff", "#e41a1c"]
cmap_oc = ListedColormap(colors)
cmap_oc.set_bad(color="#737373")

print(f"Writing overconfidence_map.png (overconfident cells: {np.sum(overconf):,})")
fig, ax = plt.subplots(figsize=(12, 5), dpi=200)
im = ax.imshow(oc_map, cmap=cmap_oc, vmin=0, vmax=1, extent=[-180, 180, -90, 90])
cbar = fig.colorbar(im, ax=ax, ticks=[0.25, 0.75], shrink=0.7)
cbar.ax.set_yticklabels(["Normal ocean", "Overconfident"])
ax.set_title(r"Overconfidence ($\sigma_Q < \sigma_P$); land masked")
fig.tight_layout()
fig.savefig(OUT_DIR / "overconfidence_map.png", dpi=200, bbox_inches="tight")
plt.show()

print(f"Done. Outputs in {OUT_DIR.relative_to(PROJECT_ROOT)}")

Writing overconfidence_map.png (overconfident cells: 107,377)
